# Project 1: Denoising Autoencoder on MNIST

**Goal:** Build a deep learning model (convolutional autoencoder) that removes noise from MNIST digit images.

**Pipeline:**
1. Load MNIST and normalize pixel values
2. Artificially add noise to the images
3. Build a convolutional autoencoder (encoder compresses, decoder reconstructs)
4. Train it to map noisy images -> clean images
5. Evaluate visually and with a quantitative metric (PSNR)
6. Save the trained model


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)


## 1. Load and preprocess the MNIST dataset

Keras ships a built-in loader that downloads MNIST automatically the first time it runs.
Pixel values are scaled to `[0, 1]` and images are reshaped to `(28, 28, 1)` (single grayscale channel) since we're using `Conv2D` layers.


In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension: (N, 28, 28) -> (N, 28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)


## 2. Add noise

We add Gaussian noise to create the "corrupted" versions of the images. The model will learn
to map `noisy_image -> clean_image`. `noise_factor` controls how much noise is added; after adding
noise we clip pixel values back to a valid `[0, 1]` range.


In [ ]:
def add_noise(images, noise_factor=0.4):
    noise = noise_factor * np.random.normal(loc=0.0, scale=1.0, size=images.shape)
    noisy_images = images + noise
    return np.clip(noisy_images, 0.0, 1.0).astype("float32")

x_train_noisy = add_noise(x_train)
x_test_noisy = add_noise(x_test)


### Visualize a few noisy vs. original images

In [ ]:
n = 8
plt.figure(figsize=(16, 4))
for i in range(n):
    # Original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Noisy
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 3. Build the convolutional autoencoder

- **Encoder:** stacks `Conv2D` + `MaxPooling2D` layers to compress the 28x28 image down into a
  smaller, dense latent representation.
- **Decoder:** mirrors the encoder using `Conv2D` + `UpSampling2D` layers to reconstruct a
  28x28 image from the latent representation.
- **Output activation:** `sigmoid`, since pixel values are normalized to `[0, 1]`.
- **Loss:** binary cross-entropy (works well for `[0, 1]`-scaled pixel reconstruction; `mse` is a
  reasonable alternative).


In [ ]:
def build_autoencoder():
    inputs = layers.Input(shape=(28, 28, 1))

    # Encoder
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2), padding="same")(x)
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    encoded = layers.MaxPooling2D((2, 2), padding="same")(x)  # (7, 7, 32) latent representation

    # Decoder
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(encoded)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.UpSampling2D((2, 2))(x)
    decoded = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same")(x)

    autoencoder = models.Model(inputs, decoded, name="denoising_autoencoder")
    autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
    return autoencoder

autoencoder = build_autoencoder()
autoencoder.summary()


## 4. Train

The model is trained to reconstruct the **clean** image from the **noisy** input
(`x_train_noisy -> x_train`). `EarlyStopping` stops training once validation loss stops improving,
and restores the best weights.


In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = autoencoder.fit(
    x_train_noisy, x_train,
    epochs=30,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
    callbacks=[early_stop],
    verbose=1,
)


### Plot training/validation loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Training history")
plt.legend()
plt.show()


## 5. Evaluate: denoise test images

Run the noisy test images through the trained autoencoder and compare
**original vs. noisy vs. denoised (reconstructed)** side by side.


In [ ]:
decoded_imgs = autoencoder.predict(x_test_noisy, verbose=0)

n = 8
plt.figure(figsize=(16, 6))
for i in range(n):
    # Original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Noisy
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    # Denoised
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(decoded_imgs[i].squeeze(), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Quantitative evaluation (PSNR)

Peak Signal-to-Noise Ratio (PSNR) gives a numeric sense of reconstruction quality — higher is
better. We compare it before denoising (noisy vs. original) and after denoising (reconstructed
vs. original).


In [ ]:
def psnr(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2, axis=(1, 2, 3))
    mse = np.clip(mse, 1e-10, None)  # avoid log(0)
    return 20 * np.log10(1.0) - 10 * np.log10(mse)

psnr_noisy = psnr(x_test, x_test_noisy).mean()
psnr_denoised = psnr(x_test, decoded_imgs).mean()

print(f"Average PSNR (noisy vs original):    {psnr_noisy:.2f} dB")
print(f"Average PSNR (denoised vs original): {psnr_denoised:.2f} dB")
print(f"Improvement: {psnr_denoised - psnr_noisy:.2f} dB")


## 7. Save the trained model

Saved in the modern Keras format (`.keras`). Reload with `tf.keras.models.load_model(...)`.


In [ ]:
autoencoder.save("denoising_autoencoder.keras")
print("Model saved to denoising_autoencoder.keras")


## Notes / possible extensions

- Try different `noise_factor` values (e.g. 0.2, 0.5, 0.7) to see how robustness changes.
- Try `mse` loss instead of `binary_crossentropy`.
- Experiment with a deeper encoder/decoder or a smaller latent bottleneck.
- Try other noise types: salt-and-pepper noise, or randomly masking patches of the image.
- Compare against a plain (non-convolutional) dense autoencoder baseline.
